# Decoder Instruction Training — Phase 2

**Role-Based Instruction Following Training**

## Key Innovation
- **Encode ONLY prompt** with CSE (system + user message)
- **Predict ONLY response** (assistant output)
- **Role markers** teach the model conversation structure

## Role Format
```
<|system|>You are a helpful assistant.<|end|>
<|user|>What is the capital of France?<|end|>
<|assistant|>The capital of France is Paris.<|end|>
```

## Training Strategy
| Component | Encoding | Loss |
|-----------|----------|------|
| System prompt | CSE ✓ | ✗ No |
| User message | CSE ✓ | ✗ No |
| Assistant response | ✗ No | ✓ Yes |

## Datasets
| Dataset | Format | Purpose |
|---------|--------|--------|
| OpenAssistant | Multi-turn | Conversations |
| Alpaca | Instruction/Response | Task following |
| Dolly | Instruction/Context/Response | Knowledge |
| HH-RLHF | Human/Assistant | Helpfulness |

## Prerequisites
- Run `decoder-training.ipynb` first (50k general training)
- This builds on those weights

---

## Cell 1: Setup Storage

In [ ]:
# Auto-detect environment and setup checkpoint storage
import os
from pathlib import Path

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        GDRIVE_BASE = Path('/content/drive/MyDrive')
        CHECKPOINT_STORAGE = GDRIVE_BASE / 'FLUX_checkpoints'
        CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)
        print(f'  ✓ Google Drive mounted')
    except:
        CHECKPOINT_STORAGE = Path('/content/checkpoints')
        CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)
        print(f'  ✓ Using local storage')
elif IS_KAGGLE:
    CHECKPOINT_STORAGE = Path('/kaggle/working/checkpoints')
    CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ Kaggle detected')
else:
    CHECKPOINT_STORAGE = Path('./checkpoints')
    CHECKPOINT_STORAGE.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ Local environment')

print(f'  Checkpoints: {CHECKPOINT_STORAGE}')
print(f'  Environment: {"Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"}')

## Cell 2: Clone/Pull Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub datasets transformers

print('  ✓ Dependencies installed')

## Cell 3: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

for phase_name in ['phase1', 'phase2', 'phase3', 'phase4', 'phase5', 'phase6', 'phase7', 'phase8', 'phase8_8', 'phase8_9', 'phase9']:
    p = PHASES_DIR / phase_name
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device, HF_REPO_ID, _resolve_hf_token

log = PhaseLogger(phase=15)  # Instruction training phase
log.separator("Decoder Instruction Training")

print('  ✓ Paths configured')

## Cell 4: Hardware & Secrets

In [ ]:
import torch
import gc

log.cell_start("Cell 4 — Hardware")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

hf_token = _resolve_hf_token()
print(f'  HF Token: {"✓" if hf_token else "✗"}')

log.cell_end("Cell 4 — Hardware", "PASS")

## Cell 5: Load Updated Flux-Apex-V1.flx

In [ ]:
from huggingface_hub import hf_hub_download
from datetime import datetime

log.cell_start("Cell 5 — Load Model")

FLX_NAME = 'Flux-Apex-V1.flx'
flx_path = CHECKPOINT_DIR / FLX_NAME

print(f'  Downloading latest {FLX_NAME}...')
try:
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=f'checkpoints/{FLX_NAME}',
        local_dir=str(ROOT),
        token=hf_token,
        force_download=True,  # Get latest from Phase 1
    )
    print(f'  ✓ Downloaded')
except Exception as e:
    print(f'  Using local: {e}')

apex = torch.load(str(flx_path), map_location='cpu', weights_only=False)
print(f'  ✓ Loaded: {FLX_NAME}')
print(f'    Version: {apex.get("version")}')

# Check for Phase 1 training
decoder_steps = apex.get('decoder', {}).get('config', {}).get('training_steps', 0)
print(f'    Decoder training steps: {decoder_steps}')

log.cell_end("Cell 5 — Load Model", "PASS")

## Cell 6: Build CSE & Decoder

In [ ]:
log.cell_start("Cell 6 — Build Models")

from wave_decoder import WaveDecoder
from cse import ContinuousSemanticEncoder

# CSE (frozen)
cse = ContinuousSemanticEncoder()
cse.load_state_dict(apex['cse']['state_dict'])
cse = cse.to(device).eval()
for p in cse.parameters():
    p.requires_grad = False
print(f'  ✓ CSE loaded (frozen)')

# Decoder (trainable)
decoder_config = apex['decoder'].get('config', {})
decoder = WaveDecoder(
    wave_dim=decoder_config.get('wave_dim', 432),
    field_features=decoder_config.get('field_features', 512),
    embed_dim=decoder_config.get('embed_dim', 256),
    hidden_dim=decoder_config.get('hidden_dim', 1024),
    num_layers=decoder_config.get('num_layers', 4),
    num_heads=decoder_config.get('num_heads', 16),
    vocab_size=decoder_config.get('vocab_size', 256),
)

decoder_state = apex['decoder']['state_dict']
cleaned_state = {k.replace('_orig_mod.', ''): v for k, v in decoder_state.items()}
decoder.load_state_dict(cleaned_state)
decoder = decoder.to(device)

n_params = sum(p.numel() for p in decoder.parameters())
print(f'  ✓ Decoder loaded: {n_params:,} parameters')

# Field context
field_state = apex['field']['state_dict']['state']
field_mean = field_state.mean(dim=(0, 1, 2))
print(f'  ✓ Field context ready')

log.cell_end("Cell 6 — Build Models", "PASS")

## Cell 7: Define Role Markers & Formatting

In [ ]:
log.cell_start("Cell 7 — Role Format")

# ─────────────────────────────────────────────
# ROLE MARKERS
# ─────────────────────────────────────────────
SYSTEM_START = "<|system|>"
SYSTEM_END = "<|end|>"
USER_START = "<|user|>"
USER_END = "<|end|>"
ASSISTANT_START = "<|assistant|>"
ASSISTANT_END = "<|end|>"

# Default system prompts (randomly selected during training)
SYSTEM_PROMPTS = [
    "You are a helpful, harmless, and honest assistant.",
    "You are a knowledgeable AI assistant. Answer questions accurately and helpfully.",
    "You are a friendly assistant. Help the user with their request.",
    "You are an AI assistant that provides clear and concise answers.",
    "You are a helpful assistant. Be informative and accurate.",
    "You are an intelligent assistant. Think step by step when needed.",
    "You are a creative assistant. Help users with writing and ideas.",
    "You are a coding assistant. Help with programming questions.",
]

def format_conversation(system_prompt: str, user_message: str, assistant_response: str) -> tuple:
    """
    Format a conversation with role markers.
    
    Returns:
        (prompt_text, response_text)
        - prompt_text: Everything the model should "see" (encode with CSE)
        - response_text: What the model should predict
    """
    prompt = f"{SYSTEM_START}{system_prompt}{SYSTEM_END}\n{USER_START}{user_message}{USER_END}\n{ASSISTANT_START}"
    response = f"{assistant_response}{ASSISTANT_END}"
    return prompt, response


def parse_instruction_format(item: dict, dataset_type: str) -> tuple:
    """
    Parse different dataset formats into (user_message, assistant_response).
    """
    if dataset_type == 'alpaca':
        instruction = item.get('instruction', '')
        input_text = item.get('input', '')
        output = item.get('output', '')
        user = f"{instruction}\n{input_text}".strip() if input_text else instruction
        return user, output
    
    elif dataset_type == 'dolly':
        instruction = item.get('instruction', '')
        context = item.get('context', '')
        response = item.get('response', '')
        user = f"{instruction}\n\nContext: {context}".strip() if context else instruction
        return user, response
    
    elif dataset_type == 'oasst':
        # OpenAssistant format
        text = item.get('text', '')
        role = item.get('role', '')
        return text, ''  # Will be handled specially
    
    elif dataset_type == 'hh_rlhf':
        # Parse Human/Assistant format
        text = item.get('chosen', '')
        if 'Human:' in text and 'Assistant:' in text:
            parts = text.split('Assistant:', 1)
            human_part = parts[0].replace('Human:', '').strip()
            assistant_part = parts[1].split('Human:')[0].strip() if len(parts) > 1 else ''
            return human_part, assistant_part
        return '', ''
    
    elif dataset_type == 'sharegpt':
        conversations = item.get('conversations', [])
        if len(conversations) >= 2:
            user_msg = next((c['value'] for c in conversations if c['from'] == 'human'), '')
            assistant_msg = next((c['value'] for c in conversations if c['from'] == 'gpt'), '')
            return user_msg, assistant_msg
        return '', ''
    
    return '', ''


# Test the formatting
test_prompt, test_response = format_conversation(
    "You are a helpful assistant.",
    "What is 2+2?",
    "2+2 equals 4."
)
print("Example Format:")
print(f"  PROMPT (encode with CSE):")
print(f"    {test_prompt}")
print(f"  RESPONSE (predict this):")
print(f"    {test_response}")

log.cell_end("Cell 7 — Role Format", "PASS")

## Cell 8: Load Instruction Datasets

In [ ]:
log.cell_start("Cell 8 — Load Datasets")

from datasets import load_dataset
import random

random.seed(42)

print("\n  Loading instruction-following datasets...")
print("  " + "="*50)

all_samples = []  # List of (prompt, response) tuples
dataset_sources = {}

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ─────────────────────────────────────────────
# 1. ALPACA (10k) — Instruction following
# ─────────────────────────────────────────────
print("\n  [1/4] Alpaca (instruction following)...")
try:
    alpaca = load_dataset("tatsu-lab/alpaca", split="train")
    count = 0
    for item in alpaca:
        user, response = parse_instruction_format(item, 'alpaca')
        if user and response and 20 < len(response) < 500:
            system = random.choice(SYSTEM_PROMPTS)
            prompt, resp = format_conversation(system, user, response)
            if len(prompt) < 400 and len(resp) < 500:
                all_samples.append((prompt, resp))
                count += 1
        if count >= 10000:
            break
    dataset_sources['alpaca'] = count
    del alpaca
    clear_mem()
    print(f"       ✓ {count} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['alpaca'] = 0

# ─────────────────────────────────────────────
# 2. DOLLY (10k) — Knowledge & instructions
# ─────────────────────────────────────────────
print("\n  [2/4] Dolly (knowledge)...")
try:
    dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
    count = 0
    for item in dolly:
        user, response = parse_instruction_format(item, 'dolly')
        if user and response and 20 < len(response) < 500:
            system = random.choice(SYSTEM_PROMPTS)
            prompt, resp = format_conversation(system, user, response)
            if len(prompt) < 400 and len(resp) < 500:
                all_samples.append((prompt, resp))
                count += 1
        if count >= 10000:
            break
    dataset_sources['dolly'] = count
    del dolly
    clear_mem()
    print(f"       ✓ {count} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['dolly'] = 0

# ─────────────────────────────────────────────
# 3. HH-RLHF (10k) — Helpful conversations
# ─────────────────────────────────────────────
print("\n  [3/4] HH-RLHF (helpfulness)...")
try:
    hh = load_dataset("Anthropic/hh-rlhf", split="train")
    count = 0
    for item in hh:
        user, response = parse_instruction_format(item, 'hh_rlhf')
        if user and response and 20 < len(response) < 500:
            system = random.choice(SYSTEM_PROMPTS)
            prompt, resp = format_conversation(system, user, response)
            if len(prompt) < 400 and len(resp) < 500:
                all_samples.append((prompt, resp))
                count += 1
        if count >= 10000:
            break
    dataset_sources['hh_rlhf'] = count
    del hh
    clear_mem()
    print(f"       ✓ {count} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    dataset_sources['hh_rlhf'] = 0

# ─────────────────────────────────────────────
# 4. CUSTOM ROLE EXAMPLES (5k) — Diverse roles
# ─────────────────────────────────────────────
print("\n  [4/4] Custom role examples...")

ROLE_EXAMPLES = [
    # Coding assistant
    ("You are a Python coding expert.", "Write a function to reverse a string", "def reverse_string(s):\n    return s[::-1]"),
    ("You are a coding assistant.", "How do I read a file in Python?", "You can use: with open('file.txt', 'r') as f: content = f.read()"),
    
    # Math tutor
    ("You are a math tutor. Explain step by step.", "What is 15% of 80?", "To find 15% of 80:\n1. Convert 15% to decimal: 15/100 = 0.15\n2. Multiply: 0.15 × 80 = 12\nAnswer: 12"),
    ("You are a helpful math assistant.", "Solve: 2x + 5 = 15", "2x + 5 = 15\n2x = 15 - 5\n2x = 10\nx = 5"),
    
    # Creative writer
    ("You are a creative writing assistant.", "Write a haiku about coding", "Lines of code flow down\nBugs hide in the semicolons\nCoffee fuels the night"),
    
    # Q&A
    ("Answer questions concisely.", "What is the capital of Japan?", "Tokyo is the capital of Japan."),
    ("You are a geography expert.", "What is the largest ocean?", "The Pacific Ocean is the largest ocean on Earth."),
    
    # Task completion
    ("You are a helpful assistant.", "Summarize: AI is transforming industries.", "AI is revolutionizing various sectors and changing how work is done."),
    ("You help with text tasks.", "Translate to formal: hey whats up", "Hello, how are you doing?"),
]

# Expand custom examples
count = 0
for _ in range(5000):
    system, user, response = random.choice(ROLE_EXAMPLES)
    prompt, resp = format_conversation(system, user, response)
    all_samples.append((prompt, resp))
    count += 1
dataset_sources['custom_roles'] = count
print(f"       ✓ {count} samples")

# Shuffle all samples
random.shuffle(all_samples)

print("\n  " + "="*50)
print("  DATASET SUMMARY:")
total = 0
for name, count in dataset_sources.items():
    print(f"    {name}: {count}")
    total += count
print(f"  {'─'*30}")
print(f"    TOTAL: {total} instruction samples")
print("  " + "="*50)

log.metric("Total samples", total)
log.cell_end("Cell 8 — Load Datasets", "PASS")

## Cell 9: Training Configuration

In [ ]:
log.cell_start("Cell 9 — Training Config")

import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Training config
GRAD_ACCUM = 4
MAX_PROMPT_LEN = 300  # Max bytes for prompt
MAX_RESPONSE_LEN = 300  # Max bytes for response
LR = 1e-4  # Lower LR for fine-tuning
CHECKPOINT_EVERY = 5000
LOG_EVERY = 500

# Optimizer
optimizer = AdamW(decoder.parameters(), lr=LR, weight_decay=0.01)

# Scheduler
total_steps = len(all_samples) // GRAD_ACCUM
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=LR/10)

print(f"  Training Configuration:")
print(f"    Total samples: {len(all_samples)}")
print(f"    Total steps: {total_steps}")
print(f"    Learning rate: {LR} (lower for fine-tuning)")
print(f"    Max prompt: {MAX_PROMPT_LEN} bytes")
print(f"    Max response: {MAX_RESPONSE_LEN} bytes")
print(f"    Checkpoint every: {CHECKPOINT_EVERY} steps")

log.cell_end("Cell 9 — Training Config", "PASS")

## Cell 10: Instruction Training Loop

In [ ]:
log.cell_start("Cell 10 — Training Loop")

from huggingface_hub import HfApi
import time

print("\n" + "="*65)
print("  INSTRUCTION TRAINING — PROMPT/RESPONSE SPLIT")
print("  Encode: prompt only | Predict: response only")
print("="*65 + "\n")

def save_checkpoint(decoder, optimizer, scheduler, step, metrics):
    checkpoint = {
        'step': step,
        'decoder_state_dict': decoder.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics,
        'timestamp': datetime.now().isoformat(),
    }
    path = CHECKPOINT_STORAGE / f'decoder_instruction_step_{step}.pt'
    torch.save(checkpoint, str(path))
    print(f"    ✓ Saved: {path.name}")
    return path

def test_instruction(decoder, cse, field_mean, system, user, max_len=100):
    """Test generation with role format."""
    decoder.eval()
    prompt, _ = format_conversation(system, user, "")
    with torch.no_grad():
        wave = cse.encode(prompt)
        wave_seq = wave.full.to(device)
        output = decoder.generate(
            wave_sequence=wave_seq,
            field_features=field_mean.to(device),
            max_length=max_len,
            temperature=0.8,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
        )
    decoder.train()
    return output.decode('utf-8', errors='replace')

# Training state
global_step = 0
accum_loss = 0.0
total_loss = 0.0
best_loss = float('inf')
start_time = time.time()

decoder.train()
optimizer.zero_grad()

for idx, (prompt, response) in enumerate(all_samples):
    try:
        # Encode prompt and response to bytes
        prompt_bytes = prompt.encode('utf-8', errors='replace')[:MAX_PROMPT_LEN]
        response_bytes = response.encode('utf-8', errors='replace')[:MAX_RESPONSE_LEN]
        
        if len(response_bytes) < 5:
            continue
        
        # TARGET: only the response bytes
        target = torch.tensor(list(response_bytes), dtype=torch.long, device=device)
        
        # ENCODE: only the prompt with CSE
        with torch.no_grad():
            wave = cse.encode(prompt[:MAX_PROMPT_LEN])
            wave_seq = wave.full.to(device)
        
        # Forward: decoder predicts response given prompt waves
        logits = decoder(
            target_bytes=target,
            wave_sequence=wave_seq,
            field_features=field_mean.to(device),
        )
        
        # Loss only on response
        loss = F.cross_entropy(logits, target)
        
        # Backward
        loss = loss / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item() * GRAD_ACCUM
        
        # Step optimizer
        if (idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            global_step += 1
            total_loss += accum_loss
            avg_loss = total_loss / global_step
            
            # Progress
            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - start_time
                rate = global_step / elapsed
                eta = (total_steps - global_step) / rate / 60
                print(f"  Step {global_step}/{total_steps}: loss={accum_loss:.4f} avg={avg_loss:.4f} lr={scheduler.get_last_lr()[0]:.2e} ETA={eta:.1f}min")
            
            # Checkpoint
            if global_step % CHECKPOINT_EVERY == 0:
                print(f"\n  ════ CHECKPOINT at step {global_step} ════")
                
                # Test with role format
                gen = test_instruction(decoder, cse, field_mean,
                    "You are a helpful assistant.",
                    "What is the capital of France?"
                )
                print(f"    Q: What is the capital of France?")
                print(f"    A: {gen}")
                
                metrics = {'step': global_step, 'loss': accum_loss, 'avg_loss': avg_loss}
                save_checkpoint(decoder, optimizer, scheduler, global_step, metrics)
                
                if accum_loss < best_loss:
                    best_loss = accum_loss
                    print(f"    ★ New best loss: {best_loss:.4f}")
                print()
            
            accum_loss = 0.0
    
    except Exception as e:
        print(f"  ⚠ Error at sample {idx}: {e}")
        continue

# Training complete
print(f"\n  ════ TRAINING COMPLETE ════")
total_time = time.time() - start_time
final_avg_loss = total_loss / max(1, global_step)

print(f"    Total steps: {global_step}")
print(f"    Final avg loss: {final_avg_loss:.4f}")
print(f"    Time: {total_time/60:.1f} minutes")

log.cell_end("Cell 10 — Training Loop", "PASS")

## Cell 11: Final Upload to HuggingFace

In [ ]:
log.cell_start("Cell 11 — Final Upload")

from huggingface_hub import HfApi

print("\n  Saving to Apex and uploading...")

# Update decoder in apex
apex['decoder']['state_dict'] = decoder.state_dict()
apex['decoder']['config']['instruction_training_steps'] = global_step
apex['metadata'] = apex.get('metadata', {})
apex['metadata']['instruction_training'] = datetime.now().isoformat()
apex['metadata']['instruction_training_samples'] = len(all_samples)

# Save locally
torch.save(apex, str(flx_path))
print(f"  ✓ Saved locally: {flx_path.name}")

# Save to storage
storage_path = CHECKPOINT_STORAGE / flx_path.name
torch.save(apex, str(storage_path))
print(f"  ✓ Saved to storage: {storage_path}")

# Upload to HuggingFace
if hf_token:
    try:
        api = HfApi(token=hf_token)
        api.upload_file(
            path_or_fileobj=str(flx_path),
            path_in_repo=f'checkpoints/{flx_path.name}',
            repo_id=HF_REPO_ID,
            commit_message=f'Instruction training complete — {global_step} steps',
        )
        print(f"  ✓ Uploaded to HuggingFace Hub")
    except Exception as e:
        print(f"  ⚠ Upload failed: {e}")

log.cell_end("Cell 11 — Final Upload", "PASS")

## Cell 12: Instruction Following Test Suite

In [ ]:
log.cell_start("Cell 12 — Test Suite")

print("\n" + "="*70)
print("  INSTRUCTION FOLLOWING TEST SUITE")
print("="*70 + "\n")

# Test cases: (system, user, expected_type, max_len)
TEST_CASES = [
    # Basic Q&A
    ("Answer questions concisely.", "What is 2+2?", "number", 30),
    ("You are a geography expert.", "What is the capital of France?", "city", 50),
    ("Answer helpfully.", "What color is the sky?", "color", 30),
    
    # Instructions
    ("You are a helpful assistant.", "Write a haiku about coding.", "poem", 100),
    ("You help with text.", "Translate to formal: hey whats up", "formal", 60),
    ("You are concise.", "Summarize: AI is changing the world in many ways.", "summary", 80),
    
    # Math
    ("You are a math tutor. Show steps.", "What is 15% of 80?", "calculation", 120),
    ("Solve math problems.", "Solve: 3x = 12", "equation", 60),
    
    # Coding
    ("You are a Python expert.", "Write a function to add two numbers.", "code", 100),
    ("You help with code.", "How do I print in Python?", "code", 50),
    
    # Conversation
    ("You are friendly and helpful.", "Hello! How are you?", "greeting", 80),
    ("You are an empathetic assistant.", "I'm feeling stressed today.", "supportive", 100),
    
    # Knowledge
    ("You are knowledgeable.", "What is photosynthesis?", "explanation", 150),
    ("Explain simply.", "What is gravity?", "explanation", 100),
    
    # Role-specific
    ("You are a chef.", "How do I make scrambled eggs?", "recipe", 150),
    ("You are a fitness coach.", "How can I start exercising?", "advice", 120),
]

decoder.eval()
results = []

print("─" * 70)
for system, user, expected_type, max_len in TEST_CASES:
    gen = test_instruction(decoder, cse, field_mean, system, user, max_len=max_len)
    
    # Basic quality check
    has_content = len(gen.strip()) > 5
    no_repetition = not any(w * 3 in gen.lower() for w in ['the', 'is', 'a', 'to'])
    status = "✓" if has_content and no_repetition else "⚠" if has_content else "✗"
    
    results.append(status)
    
    print(f"\n  [{expected_type.upper()}]")
    print(f"  System: {system[:40]}..." if len(system) > 40 else f"  System: {system}")
    print(f"  User: {user}")
    print(f"  Response: {gen}")
    print(f"  Status: {status}")
    print("─" * 70)

# Summary
success = results.count('✓')
warning = results.count('⚠')
fail = results.count('✗')

print(f"\n  RESULTS: {success}✓ {warning}⚠ {fail}✗ out of {len(results)} tests")
print(f"  Success rate: {100*success/len(results):.0f}%")

log.cell_end("Cell 12 — Test Suite", "PASS")

## Cell 13: Interactive Testing

In [ ]:
# Interactive testing cell - run multiple times with different inputs

# Change these to test different scenarios:
SYSTEM = "You are a helpful, friendly assistant."
USER = "What is the meaning of life?"
MAX_LEN = 150

print(f"System: {SYSTEM}")
print(f"User: {USER}")
print(f"─" * 50)

response = test_instruction(decoder, cse, field_mean, SYSTEM, USER, max_len=MAX_LEN)
print(f"Assistant: {response}")

## Cell 14: Summary

In [ ]:
log.separator("Instruction Training Complete")

print("""
╔════════════════════════════════════════════════════════════════════╗
║         INSTRUCTION TRAINING COMPLETE                              ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  KEY INNOVATIONS:                                                  ║
║  ✓ Prompt/Response split training                                 ║
║  ✓ Role markers (<|system|>, <|user|>, <|assistant|>)            ║
║  ✓ Encode prompt only → Predict response only                     ║
║                                                                    ║
║  TRAINING DATA:                                                    ║
║  ├── Alpaca (instruction following)                               ║
║  ├── Dolly (knowledge & tasks)                                    ║
║  ├── HH-RLHF (helpfulness)                                        ║
║  └── Custom roles (diverse scenarios)                             ║
║                                                                    ║
║  MODEL CAPABILITIES:                                               ║
║  • Answer questions based on prompt                               ║
║  • Follow instructions                                             ║
║  • Assume different roles                                          ║
║  • Multi-turn potential                                            ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

print(f"\nFinal Statistics:")
print(f"  Instruction samples: {len(all_samples)}")
print(f"  Total steps: {global_step}")
print(f"  Final avg loss: {final_avg_loss:.4f}")
print(f"  Training time: {total_time/60:.1f} minutes")

log.success("Instruction training complete!")